In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict, Annotated
from langgraph.types import Send
import operator

In [ ]:
from typing import Union

class State(TypedDict):
    words: list[str]
    output: Annotated[list[dict[str, Union[str, int]]], operator.add]

graph_builder = StateGraph(State)

In [ ]:
def node_one(state: State):    
    print(f"I want to count {len(state["words"])} words in my state.")

def node_two(word: str):
    return {
        "output": [{
            "word": word,
            "letters": len(word)    
        }]
    }


In [ ]:
graph_builder.add_node("node_one", node_one)
graph_builder.add_node("node_two", node_two)

def dispatcher(state: State):
    return [Send("node_two", word) for word in state["words"]]

    #words = state["words"]
    #sends = []
    #for word in words:
    #    sends.append(Send("node_two", word))
    #return sends

graph_builder.add_edge(START, "node_one")
graph_builder.add_conditional_edges("node_one", dispatcher, ['node_two'])
graph_builder.add_edge("node_two", END)

In [ ]:
graph = graph_builder.compile()

graph.invoke({
    "words": ['hello', 'world', 'how', 'are', 'you', 'doing']
})

In [ ]:
graph